In [1]:
from __future__ import annotations

import os

import evaluate
import numpy as np
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from dataset_loader import SummarizationDatasetLoader

In [2]:
MODEL_NAME  = "VietAI/vit5-base-vietnews-summarization"
DATA_PATH   = "../../data/processed/df_processed2603.jsonl"
OUTPUT_DIR  = "../models/results_vit5"
CACHE_DIR   = "../../data/tokenized_cache"
HF_TOKEN    = os.getenv("HF_TOKEN")

In [3]:
loader = SummarizationDatasetLoader(
    model_name=MODEL_NAME,
    max_input_length=1024,
    max_target_length=256,
    cache_dir=CACHE_DIR,
    hf_token=HF_TOKEN,
)
tokenized_datasets = loader.get_ready_dataset(file_path=DATA_PATH)
tokenizer = loader.tokenizer

2026-03-26 16:11:33,208 [INFO] Loading tokenizer: VietAI/vit5-base-vietnews-summarization
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
2026-03-26 16:11:33,669 [INFO] Loading tokenized dataset from cache: ../../data/tokenized_cache


In [4]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, token=HF_TOKEN)

# Gradient checkpointing: trades ~20% speed for ~40% less VRAM - worth it on 6 GB
model.gradient_checkpointing_enable()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")
model.to(device)

/home/hao0107/miniconda3/envs/text_sum_venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Training on: cuda


T5ForConditionalGeneration(
  (shared): Embedding(36096, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(36096, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [5]:
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 (loss mask) with pad token before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip leading/trailing whitespace per sample
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    return {k: round(v * 100, 4) for k, v in result.items()}

In [6]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,   # consistent with preprocess_function masking
)

In [7]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # --- Evaluation & checkpointing ---
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",   # track rougeL for best-model selection
    greater_is_better=True,

    # --- Batch & accumulation (effective batch = 4 * 4 = 16) ---
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    # --- Learning rate (cosine schedule with warm-up) ---
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,            # 5% of total steps for warm-up

    # --- Regularisation ---
    weight_decay=0.01,

    # --- Epochs & checkpoints ---
    num_train_epochs=10,
    save_total_limit=2,           # keep only the 2 best checkpoints

    # --- Generation (needed for ROUGE during eval) ---
    predict_with_generate=True,
    generation_max_length=128,

    # --- Mixed precision (fp16 on CUDA, skip on CPU) ---
    fp16=torch.cuda.is_available(),

    # --- Logging ---
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",             # change to "tensorboard" or "wandb" if wanted

    push_to_hub=False,
)

In [8]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        # Stop early if rougeL does not improve for 3 consecutive epochs
        EarlyStoppingCallback(early_stopping_patience=3)
    ],
)

In [9]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
0,1.471000,1.205443,71.106100,42.825500,47.884900,47.883800
1,1.161200,1.105126,72.286800,44.178200,48.691300,48.663400
2,0.927600,1.090820,73.267400,45.241100,49.300200,49.283500
4,0.633400,1.143844,73.275600,45.879000,49.700300,49.694100
5,0.520700,1.176468,73.392000,45.620900,49.344200,49.294900
6,0.463300,1.230674,73.578100,46.019200,49.969200,49.955500
8,0.382800,1.272541,73.622500,45.969600,49.678100,49.685200
9,0.371100,1.276177,73.695800,46.109000,49.639400,49.637600


2026-03-26 16:25:37,175 [INFO] Using default tokenizer.
2026-03-26 16:38:57,058 [INFO] Using default tokenizer.
2026-03-26 16:52:12,594 [INFO] Using default tokenizer.
2026-03-26 17:05:29,580 [INFO] Using default tokenizer.
2026-03-26 17:18:39,274 [INFO] Using default tokenizer.
2026-03-26 17:31:47,470 [INFO] Using default tokenizer.
2026-03-26 17:44:57,795 [INFO] Using default tokenizer.
2026-03-26 17:58:12,711 [INFO] Using default tokenizer.
2026-03-26 18:12:18,603 [INFO] Using default tokenizer.
2026-03-26 18:25:44,744 [INFO] Using default tokenizer.
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1730, training_loss=0.762680357453451, metrics={'train_runtime': 8069.8842, 'train_samples_per_second': 3.431, 'train_steps_per_second': 0.214, 'total_flos': 1.683518680576512e+16, 'train_loss': 0.762680357453451, 'epoch': 9.985569985569985})

In [10]:
best_path = os.path.join(OUTPUT_DIR, "vit5_model_v2")
trainer.save_model(best_path)
tokenizer.save_pretrained(best_path)
print(f"--- Training complete. Best model saved to: {best_path} ---")

--- Training complete. Best model saved to: ../models/results_vit5/vit5_model_v2 ---
